# Stage Part A output to the SDP landing zone

Copies only the **top-level final `*.json`** documents produced by Part A into the landing volume consumed by the Document Intelligence SDP pipeline. Skips the `chunks/` working directory, `extraction_log.jsonl`, and any file already staged (idempotent — safe to re-run).

In [0]:
dbutils.widgets.text("extracted_path", "/Volumes/ext_cat/doc_intel/extracted", "Part A output path")
dbutils.widgets.text("landing_path", "/Volumes/ext_cat/doc_intel/landing", "SDP landing zone")

EXTRACTED_PATH = dbutils.widgets.get("extracted_path").rstrip("/")
LANDING_PATH = dbutils.widgets.get("landing_path").rstrip("/")
print(f"Staging {EXTRACTED_PATH} -> {LANDING_PATH}")

In [0]:
landing_names = {f.name for f in dbutils.fs.ls(LANDING_PATH)}
copied, skipped = [], []

for f in dbutils.fs.ls(EXTRACTED_PATH):
    name = f.name
    if f.path.endswith("/") or not name.endswith(".json") or name == "extraction_log.jsonl":
        continue  # skip chunks/, sidecar logs, non-JSON
    if name in landing_names:
        skipped.append(name)  # already staged (idempotent)
        continue
    dbutils.fs.cp(f"{EXTRACTED_PATH}/{name}", f"{LANDING_PATH}/{name}")
    copied.append(name)

print(f"Copied {len(copied)} new document(s), skipped {len(skipped)} already staged.")
if copied:
    print("New in landing:", ", ".join(sorted(copied)))